# 80 — Blind-A responder swap: Gemini-generated `predicted_response` -> submission zip

Takes the existing Blind-A prediction (whose `predicted_track_ids` came from the
validated retrieval+rerank pipeline, e.g. config 194 via colab/41) and regenerates
ONLY `predicted_response` with the Gemini API, then packages the CodaBench zip.
`predicted_track_ids` are left untouched -> nDCG/diversity axes unchanged; this only
moves the LLM axis (0.30 of the composite).

No GPU needed (no local responder, no retrieval). Prereqs: `GEMINI_API_KEY` in Colab
secrets, and an existing Blind-A `predicted_track_ids` prediction.json (from colab/41
or on Drive). top_n=1 by default (v5-kto's winning setting; not assumed optimal for
Gemini -- A/B 1 vs 3 on dev later via nb79's judge).

Rules note: external LLM APIs aren't banned, but final code must be uploaded
(due 2026-07-09) and the judge family is Gemini (self-preference risk). See
project_responder_topn_ab_status_2026_06_04 memory.


In [ ]:
# 1) Setup — clone branch + Gemini key + Drive + light deps (no GPU).
import os
os.environ['USE_FLAX'] = '0'; os.environ['USE_TF'] = '0'
from google.colab import userdata, drive
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')          # add this Colab secret first
os.environ.setdefault('GEMINI_RESPONDER_MODEL', 'gemini-2.5-flash')    # 1.5-flash is retired
drive.mount('/content/drive', force_remount=False)
BRANCH = 'recall-union-lgbm'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026
!pip install -q -U google-generativeai 'datasets' 'pandas<3.0'
print('setup done | GEMINI key present:', bool(os.environ.get('GEMINI_API_KEY')))


In [ ]:
# 2) CONFIG — locate the existing Blind-A prediction (predicted_track_ids to KEEP).
import os, json
from pathlib import Path

TID = '194-union-sasrec-lgbm-cleanfull-v5kto-blindA'   # the pipeline run whose tracks we keep
TOP_N = 1                                              # tracks the Gemini responder sees/explains
BLIND_DATASET = 'talkpl-ai/TalkPlayData-Challenge-Blind-A'

PRED_IN  = f'/content/recsys2026/music-crs-baselines/exp/inference/blindset_A/{TID}.json'
PRED_OUT = f'/content/recsys2026/music-crs-baselines/exp/inference/blindset_A/{TID}_gemini.json'

# Not in the fresh clone -> pull from Drive (where colab/41 saved Blind-A runs).
# EDIT DRIVE_PRED if your predicted_track_ids json lives elsewhere.
DRIVE_PRED = f'/content/drive/MyDrive/blindset_runs/{TID}.json'
os.makedirs(os.path.dirname(PRED_IN), exist_ok=True)
if not Path(PRED_IN).exists():
    if Path(DRIVE_PRED).exists():
        import shutil; shutil.copy(DRIVE_PRED, PRED_IN)
        print('copied prediction from Drive:', DRIVE_PRED)
    else:
        raise FileNotFoundError(
            f'No Blind-A prediction at\n  {PRED_IN}\nor\n  {DRIVE_PRED}\n'
            'Run colab/41_run_blindset_A.ipynb (retrieval+rerank, GPU) first, '
            'or set DRIVE_PRED to your existing predicted_track_ids prediction.json.')

rows = json.load(open(PRED_IN))
assert all('predicted_track_ids' in r for r in rows), 'input must carry predicted_track_ids'
print(f'{len(rows)} Blind-A rows (expect 80) | keeping predicted_track_ids, regenerating responses')


In [ ]:
# 3) Regenerate predicted_response over Blind-A via Gemini (track_ids untouched).
#    On any per-row API failure the script keeps that row's original response.
!cd /content/recsys2026 && python -u scripts/gemini_responder.py \
    --pred {PRED_IN} --out {PRED_OUT} \
    --dataset {BLIND_DATASET} --top-n {TOP_N} --sleep 0.2
import json
out = json.load(open(PRED_OUT))
print(f'\nrows out: {len(out)}')
print('sample response:\n', out[0]['predicted_response'][:500])


In [ ]:
# 4) Validate schema (blindA) + package the CodaBench zip (root = prediction.json).
from datetime import date
import os, sys, zipfile, shutil
sys.path.insert(0, '/content/recsys2026/scripts')
from validate_prediction import load_prediction, validate_schema, package_zip

predictions = load_prediction(PRED_OUT)
errors = validate_schema(predictions, 'blindA')
if errors:
    print('Schema validation FAILED:')
    for e in errors[:20]: print('  -', e)
    raise SystemExit('Refusing to package — fix and rerun.')
print(f'schema OK ({len(predictions)} rows for blindA — expected 80)')

ZIP_PATH = f'/content/recsys2026/data/submissions/blindset_A_{date.today().isoformat()}_{TID}_gemini.zip'
os.makedirs(os.path.dirname(ZIP_PATH), exist_ok=True)
out_zip = package_zip(PRED_OUT, ZIP_PATH)
with zipfile.ZipFile(out_zip) as zf:
    members = zf.namelist()
assert members == ['prediction.json'], f'wrong zip layout: {members}'
print('packaged ->', out_zip, '| contains', members)

drive_zip = f'/content/drive/MyDrive/blindset_runs/{os.path.basename(ZIP_PATH)}'
os.makedirs(os.path.dirname(drive_zip), exist_ok=True)
shutil.copy(out_zip, drive_zip)
print('Drive copy ->', drive_zip)
print('\nUpload this zip to CodaBench.')
